### Deep Learning & Neural Networks — Diabetes Prediction with TensorFlow/Keras
**Week 6 — Generative AI Learning Journey**

---

**Goal**: Build a binary classification neural network to predict diabetes (yes/no).
**Dataset**: Pima Indians Diabetes — 768 patients, 8 medical features.
**Framework**: TensorFlow / Keras (Google's deep learning library).

---

**Java → Deep Learning Quick Glossary**

| Concept | Java Analogy | DL Meaning |
|---|---|---|
| Neural Network | Chain of nested method calls | Layered system that learns patterns |
| Layer | A method in a pipeline | Group of neurons that transform data |
| Neuron | Variable with a learned coefficient | Weighted sum + activation unit |
| Weight | Field value that gets updated | Multiplier the model learns per input |
| Bias | Constant offset (`+= c`) | Additional learned term per neuron |
| Activation | `Math.max()` / `if-else` | Non-linear function on neuron output |
| Epoch | Full `for` loop over all training data | One complete pass through training set |
| Batch | Processing a `List` in chunks | Subset of rows per weight update |
| Loss | Error score / cost function | How wrong the predictions currently are |
| Optimizer | Learning strategy | Algorithm that minimizes loss (e.g. Adam) |
| Backpropagation | Debugging backwards through a stack | Gradient flows back to update weights |
| Forward Pass | Calling a method chain | Input flowing through layers → prediction |
| Overfitting | Unit test passes, integration fails | Memorizes training, fails on new data |

---

## Concept Classification — ANN / MLP (Artificial Neural Network / Multi-Layer Perceptron)

This notebook covers the **foundational deep learning architecture** — feedforward neural networks built from Dense (fully connected) layers.

```
Deep Learning
├── ANN / MLP  ← YOU ARE HERE
│     Feedforward networks · Dense layers · backprop · classification & regression
├── CNN        → Convolutional Neural Networks (images, spatial data)
├── RNN / LSTM → Recurrent Neural Networks (sequences, time series, NLP)
└── Transformers → Attention-based models (BERT, GPT, LLMs)
```

**What this notebook covers:**
- Building a feedforward ANN with TensorFlow / Keras (`Sequential`, `Dense`, `InputLayer`)
- Binary classification on the Pima Indians Diabetes dataset (768 patients, 8 features)
- Activation functions — ReLU (hidden layers), Sigmoid (output)
- Loss: Binary Cross-Entropy · Optimizer: Adam · Metric: Accuracy
- Epoch/batch training loop, validation monitoring, overfitting detection
- Inference with `predict()` vs scoring with `evaluate()`

**Companion documents:**
- [`ANN_Comprehensive_Guide.md`](ANN_Comprehensive_Guide.md) — Full ANN/MLP theory + 17 interview questions (beginner → advanced)
- [`Deep_Learning_Clean.md`](Deep_Learning_Clean.md) — Master reference covering all ML + DL concepts
- [`RNN_Comprehensive_Guide.md`](RNN_Comprehensive_Guide.md) — RNN / LSTM / GRU deep dive + interview questions

---

In [ ]:
# TensorFlow = Google's deep learning framework
# Keras = TensorFlow's high-level API (like Spring Boot is to Java EE)
# PyTorch (Meta) is the other major framework — less boilerplate, popular in research

!pip install tensorflow

#### Deep Learning — Core Concepts

**What is a Neural Network?**
A chain of mathematical transformations that learns patterns from data.

Java analogy:
```java
// Each layer is a transformation; the network learns the best weights
double output = sigmoid(relu(relu(relu(matMul(input, weights) + bias))));
```

---

**The 5 Core Building Blocks**

**1. Neurons** — each computes: `output = activation(Σ(inputᵢ × weightᵢ) + bias)`

**2. Layers**
- Input Layer: receives raw features (like constructor parameters)
- Hidden Layers: intermediate transformations (like private helper methods)
- Output Layer: final prediction (like the `return` statement)

**3. Weights & Biases** — learned during training. Weights = multipliers; bias = constant offset.

**4. Activation Functions** — introduce non-linearity (without them, stacking layers = one linear transform)

| Function | Formula | Use Case |
|---|---|---|
| ReLU | `max(0, x)` | Hidden layers — fast, avoids vanishing gradient |
| Sigmoid | `1 / (1 + e⁻ˣ)` | Binary output layer — probability [0, 1] |
| Softmax | `eˣᵢ / Σeˣ` | Multi-class output |
| Tanh | `(eˣ - e⁻ˣ) / (eˣ + e⁻ˣ)` | RNNs, some hidden layers |

**5. Forward Pass vs Backward Pass**
- Forward: input → layers → prediction (like calling a method)
- Backward (Backpropagation): error → gradient flows back → weights updated (like debugging + fixing)

---

> **Interview Q: What is a neural network?**
> A computational model of layers of connected neurons. Each neuron computes a weighted sum of inputs plus bias, then applies an activation function. The network learns by adjusting weights via backpropagation and gradient descent to minimize prediction error.

In [ ]:
import numpy as np      # Fast numerical arrays — Java: Apache Commons Math / double[][]
import pandas as pd     # Tabular data — Java: DataFrame / ResultSet

# Keras high-level API — Sequential is a linear layer pipeline (Builder pattern in Java)
from tensorflow.keras.models import Sequential
# Dense = fully-connected layer (every neuron → every neuron in next layer)
# InputLayer = declares the input shape (like a method signature)
from tensorflow.keras.layers import Dense, InputLayer

from sklearn.model_selection import train_test_split  # splits data into train/test

print("Libraries imported successfully!")

#### Dataset — Pima Indians Diabetes

Medical records from Pima Indian women aged 21+.

Java analogy — each row is an instance of:
```java
class Patient {
    int    pregnancies;               // number of pregnancies
    double glucose;                   // plasma glucose (mg/dL) — key indicator
    double bloodPressure;             // diastolic BP (mm Hg)
    double skinThickness;             // triceps skinfold (mm)
    double insulin;                   // 2-hour serum insulin
    double bmi;                       // body mass index
    double diabetesPedigreeFunction;  // genetic risk score
    int    age;
    boolean isDiabetic;               // TARGET: 0 = No, 1 = Yes
}
```

| Feature | High Value Risk? |
|---|---|
| Glucose | **Strong** |
| BMI | **Strong** |
| DiabetesPedigreeFunction | High |
| Age | Moderate |
| Insulin | Moderate |

**768 patients · Binary classification (0 = No Diabetes, 1 = Diabetes)**

In [ ]:
# pd.read_csv() → reads CSV into DataFrame
# Java: like SELECT * FROM patients into a List<Patient>
df = pd.read_csv("pima-indians-diabetes.csv")

print(f"Dataset shape: {df.shape}")   # (768, 9) — 768 rows, 9 columns
df.head()                              # show first 5 rows

In [ ]:
# Exploratory Data Analysis — always inspect data before modelling
# Java analogy: input validation / assertions before processing

print("Column Names:", df.columns.tolist())

print("\nData Types:")
print(df.dtypes)
# int64   → Java long  |  float64 → Java double

print("\nBasic Statistics:")
# Key things to check:
#   min = 0 for Glucose/BP → biologically impossible → likely missing values encoded as 0
#   large std → high spread → may need feature scaling
df.describe()

#### Splitting Features (X) and Target (y)

- **X (Features)** — the columns the model uses as input evidence
- **y (Target)** — the column the model must predict

Java analogy:
```java
// X = input parameters  |  y = the return value the model learns
double predict(double[] X) { return y; }
```

The model must never "see" y during training — it must figure it out from X alone.
(Like a student solving problems without peeking at the answer key.)

`.values` converts a Pandas DataFrame to a raw NumPy array (fast `double[][]`).

In [ ]:
# Drop "Outcome" column → remaining 8 columns are our features
# .values converts DataFrame → NumPy 2D array (768 rows × 8 columns)
X = df.drop(columns=["Outcome"]).values

print(f"Features shape  : {X.shape}")          # (768, 8)
print(f"Data type       : {X.dtype}")           # float64
print(f"Feature columns : {df.drop(columns=['Outcome']).columns.tolist()}")
print(f"\nSample patient  : {X[0]}")            # first patient's 8 values

In [ ]:
# Target: 0 = No Diabetes, 1 = Diabetes
# Java: int[] y = extractColumn(dataset, "Outcome");
y = df["Outcome"].values

print(f"Target shape   : {y.shape}")            # (768,) — 1D, one label per patient
print(f"Unique values  : {np.unique(y)}")       # [0 1]

# CLASS DISTRIBUTION — important to check for imbalance
# If 90% non-diabetic, a model always predicting "No" gets 90% accuracy — but is useless
diabetic     = sum(y == 1)
non_diabetic = sum(y == 0)
print(f"\nClass Distribution:")
print(f"  Non-Diabetic (0): {non_diabetic}  ({non_diabetic/len(y)*100:.1f}%)")
print(f"  Diabetic     (1): {diabetic}  ({diabetic/len(y)*100:.1f}%)")
print(f"\n  Mildly imbalanced — consider stratify=y and class_weight in model.fit()")

#### Train / Test Split

| Set | Size | Purpose |
|---|---|---|
| Training set | ~80% | Model *learns* from this — weights updated |
| Test set | ~20% | Model *evaluated* on this — never seen during training |

Java analogy: splitting your test suite — some tests implement/refine the logic, some validate independently.

- `random_state=42` — fixed seed for reproducibility (`new Random(42)` in Java)
- `stratify=y` (recommended) — preserves class proportions in both splits
- **Data leakage**: if model trains on test data → artificially inflated accuracy → useless in production

> **Interview Q:** Why do we need a test set?
> To measure how well the model **generalizes** to unseen data. Without it, we'd only know if it memorized training examples (overfitting), not if it works in production.

In [ ]:
# 80% training / 20% test
# random_state=42 → reproducible split (same result every run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
    # Recommended addition: stratify=y
)

print(f"Training samples : {len(X_train)}")    # ~614
print(f"Test samples     : {len(X_test)}")     # ~154
print(f"Train X shape    : {X_train.shape}")   # (614, 8)
print(f"Test X shape     : {X_test.shape}")    # (154, 8)

In [ ]:
# BATCH SIZE — how many samples to process before updating weights
#
# Java analogy:
#   for (int i = 0; i < data.size(); i += batchSize) {
#       updateWeights(data.subList(i, i + batchSize));
#   }
#
# Three gradient descent strategies:
#   Batch GD      → all data at once  (stable but slow, memory-heavy)
#   Mini-Batch GD → small chunk       (balanced — most common) ← THIS
#   Stochastic GD → 1 sample at a time (fast updates, very noisy)
#
# batch_size=10: smaller batch → noisier but more frequent updates

train_size = len(X_train)
test_size  = len(X_test)
batch_size = 10

steps_per_epoch = train_size // batch_size   # weight updates per epoch

print(f"Train size       : {train_size}")
print(f"Test size        : {test_size}")
print(f"Batch size       : {batch_size}")
print(f"Steps per epoch  : {steps_per_epoch}  ({train_size} / {batch_size})")

#### Neural Network Architecture

```
Input (8 patient features)
        │
 [InputLayer: 8]         declares: "I expect 8 numbers per sample"
        │
 [Dense: 32, ReLU]       Hidden Layer 1 — learns broad patterns
        │                32×8 = 256 weights + 32 biases = 288 params
 [Dense: 16, ReLU]       Hidden Layer 2 — refines patterns
        │                16×32 = 512 weights + 16 biases = 528 params
 [Dense:  8, ReLU]       Hidden Layer 3 — fine-grained patterns
        │                8×16 = 128 weights + 8 biases = 136 params
 [Dense:  1, Sigmoid]    Output — probability of Diabetes [0.0 → 1.0]
        │                1×8 = 8 weights + 1 bias = 9 params
 Probability >= 0.5 → Diabetic | < 0.5 → Non-Diabetic
```

**Why Sequential?** Layers in a straight line — output of one → input of next (Java filter chain).

**Why funnel shape (32→16→8)?** Each layer compresses the representation — wide early layers capture many patterns, narrow later layers distill the most important ones.

**Why ReLU in hidden layers?** `max(0, x)` — fast, avoids vanishing gradient. Java: `Math.max(0, x)`.

**Why Sigmoid in the output layer only?** Maps output to [0,1] = probability. Sigmoid causes vanishing gradients in deep hidden layers, so only use it at the output.

> **Interview Q: What is the vanishing gradient problem?**
> Sigmoid/tanh derivatives are < 1, so gradients shrink exponentially as they backpropagate through many layers — early layers stop learning. ReLU avoids this (gradient = 0 or 1, no shrinking). This discovery enabled modern deep learning.

In [ ]:
# Sequential = linear layer pipeline (Builder pattern in Java)
model = Sequential([

    # INPUT: declares expected input shape — no computation here
    # Java: void predict(double[8] input)
    InputLayer(input_shape=(8,)),

    # HIDDEN LAYER 1: 32 neurons, ReLU
    # output = max(0, weights·input + bias)  for each neuron
    # Parameters: 32×8 + 32 = 288
    Dense(32, activation="relu"),

    # HIDDEN LAYER 2: 16 neurons, ReLU
    # Parameters: 16×32 + 16 = 528
    Dense(16, activation="relu"),

    # HIDDEN LAYER 3: 8 neurons, ReLU
    # Parameters: 8×16 + 8 = 136
    Dense(8, activation="relu"),

    # OUTPUT: 1 neuron, Sigmoid → probability in [0.0, 1.0]
    # >= 0.5 → Diabetic | < 0.5 → Non-Diabetic
    # Parameters: 1×8 + 1 = 9
    Dense(1, activation="sigmoid")
])

# Total learnable parameters: 288 + 528 + 136 + 9 = 961
model.summary()

#### Compiling the Model — Setting the Learning Strategy

Three components to configure before training:

| Component | Our Choice | What It Does |
|---|---|---|
| **Loss function** | `binary_crossentropy` | Measures how wrong predictions are (what we minimize) |
| **Optimizer** | `adam` | Algorithm that adjusts weights to reduce loss |
| **Metrics** | `accuracy` | What we display each epoch (does NOT affect training) |

**Loss function by task type:**
- Binary classification (0/1) → `binary_crossentropy`
- Multi-class → `categorical_crossentropy`
- Regression (predict a number) → `mean_squared_error`

**Binary Cross-Entropy** formula: `-[y·log(p) + (1-y)·log(1-p)]`
Confident wrong predictions are penalized heavily; confident correct ones score near 0.

**Adam optimizer** — adapts learning rate per weight automatically. Best default choice.

> **Interview Q: What is gradient descent?**
> An algorithm that minimizes loss by iteratively moving weights in the direction of the negative gradient. Learning rate controls step size. Too large → training diverges. Too small → very slow. Adam adapts the rate per parameter automatically — more robust than plain SGD.

In [ ]:
model.compile(
    loss="binary_crossentropy",   # standard for binary 0/1 classification
    optimizer="adam",             # adaptive learning rate — most popular optimizer
    metrics=["accuracy"]          # display % correct each epoch (monitoring only)
)

print("Model compiled!")
print("Loss      : Binary Cross-Entropy")
print("Optimizer : Adam (adaptive learning rate)")
print("Metric    : Accuracy")

#### Training — The Learning Loop

What happens inside `model.fit()` each epoch:

```
FOR each epoch (1 → 150):
    FOR each batch of 10 samples:
        1. Forward pass:  batch → layers → prediction
        2. Compute loss:  how wrong is the prediction?
        3. Backpropagation: compute gradient of loss for every weight
        4. Adam update:   adjust all weights slightly to reduce loss
    END batch
    REPORT: loss, accuracy, val_loss, val_accuracy
END epoch
```

Java pseudocode:
```java
for (int epoch = 0; epoch < 150; epoch++) {
    for (double[][] batch : getBatches(X_train, 10)) {
        double[] preds = forwardPass(batch);
        double   loss  = binaryCrossEntropy(preds);
        double[] grads = backpropagate(loss);
        adam.updateWeights(model.weights, grads);
    }
}
```

**`validation_split=0.1`** — reserves 10% of training data to monitor overfitting (not used for weight updates).

| Data portion | Used for |
|---|---|
| 72% training | Weight updates — model learns from this |
| 8% validation | Overfitting check each epoch |
| 20% test | Final evaluation — untouched until the end |

> **Interview Q: What is overfitting?**
> When a model performs well on training data but poorly on new data — it memorized examples instead of learning generalizable patterns. Signs: train accuracy >> val accuracy, val loss rising. Fixes: Dropout, L1/L2 regularization, early stopping, more data, simpler architecture.

In [ ]:
# model.fit() runs the full training loop:
#   forward pass → loss → backprop → weight update, repeated for every epoch
history = model.fit(
    X_train, y_train,
    epochs=150,             # full passes through training data
    batch_size=batch_size,  # 10 samples per weight update
    validation_split=0.1,   # 10% held out for overfitting monitoring
    verbose=1               # 0=silent | 1=progress bar | 2=one line per epoch
)

print("\nTraining complete!")
print(f"Final Train Accuracy : {history.history['accuracy'][-1]*100:.2f}%")
print(f"Final Val Accuracy   : {history.history['val_accuracy'][-1]*100:.2f}%")

#### Model Evaluation — Measuring Real-World Performance

The test set has never been seen at any point — it gives an honest estimate of production performance.

**Metrics explained:**

| Metric | Goal | Notes |
|---|---|---|
| Loss | Lower is better | Cross-entropy on test data |
| Accuracy | Higher is better | Can be misleading if classes are imbalanced |
| Precision | TP / (TP+FP) | Use when false positives are costly |
| Recall | TP / (TP+FN) | Use when false negatives are costly ← medical |
| F1-Score | 2·P·R / (P+R) | Balanced metric for imbalanced datasets |

For diabetes detection: **recall is critical** — missing a true diabetic (false negative) is worse than a false alarm.

> **Interview Q: Why can accuracy be misleading?**
> If 65% of patients are non-diabetic, a model that always predicts "No" gets 65% accuracy — but catches zero real diabetics. Always check class distribution and use precision/recall/F1 for imbalanced datasets.

In [ ]:
# model.evaluate() runs forward pass only — NO weight updates
# Java analogy: scoring method on a held-out test dataset
loss, accuracy = model.evaluate(X_test, y_test, batch_size=10, verbose=1)

print(f"\nTest Loss     : {loss:.4f}")
print(f"Test Accuracy : {accuracy*100:.2f}%")

# Overfitting check
final_train_acc = history.history['accuracy'][-1]
gap = abs(final_train_acc - accuracy)
print(f"\nTrain Accuracy : {final_train_acc*100:.2f}%")
print(f"Test Accuracy  : {accuracy*100:.2f}%")
print(f"Gap            : {gap*100:.1f}%  — {'possible overfitting' if gap > 0.08 else 'healthy generalization'}")

print(f"\nKnown improvements:")
print(f"  1. Feature scaling (StandardScaler) — neural nets are scale-sensitive")
print(f"  2. Replace 0-values in Glucose/BP with column median")
print(f"  3. Add Dropout(0.2) after hidden layers")
print(f"  4. Use class_weight in model.fit() for imbalance")

#### Making Predictions — Inference Mode

`model.predict()` runs a forward pass only — no backprop, no weight changes.

Output is a **probability** from the Sigmoid layer:
```
0.82 → >= 0.5 → Diabetic
0.23 → <  0.5 → Non-Diabetic
0.51 → >= 0.5 → Diabetic (barely — model is uncertain)
```

Java analogy:
```java
double probability = model.predict(patientFeatures);
String diagnosis   = probability >= 0.5 ? "Diabetic" : "Non-Diabetic";
```

**Decision threshold is tunable:**
- Lower (e.g. 0.3) → more sensitive — catches more true diabetics, more false alarms
- Higher (e.g. 0.7) → more specific — fewer false alarms, misses more cases

> **Interview Q: What is a decision boundary?**
> The threshold separating predicted classes. Default is 0.5 for Sigmoid output. Adjusting it trades off precision vs recall based on domain requirements.

In [ ]:
# model.predict() → forward pass only, returns probabilities (n_samples, 1)
predictions = model.predict(X_test)

header = (
    f"{'#':<4}  {'Preg':>5} {'Gluc':>5} {'BP':>5} {'Skin':>5} "
    f"{'Ins':>6} {'BMI':>6} {'DPF':>7} {'Age':>4}  "
    f"{'Prob':>8}  {'Actual':>12}  {'Predicted':>12}  {'OK?':>4}"
)
print(header)
print("-" * len(header))

for i in range(5):
    row  = X_test[i]
    prob = predictions[i][0]
    pred_label   = "Diabetic"   if prob >= 0.5    else "Non-Diabetic"
    actual_label = "Diabetic"   if y_test[i] == 1 else "Non-Diabetic"
    match        = "OK" if pred_label == actual_label else "WRONG"

    print(
        f"{i:<4}  {row[0]:>5.0f} {row[1]:>5.0f} {row[2]:>5.0f} {row[3]:>5.0f} "
        f"{row[4]:>6.0f} {row[5]:>6.1f} {row[6]:>7.3f} {row[7]:>4.0f}  "
        f"{prob:>8.4f}  {actual_label:>12}  {pred_label:>12}  {match:>5}"
    )
print(f"\nProbability >= 0.5 = Diabetic  |  < 0.5 = Non-Diabetic")
print(f"Values near 0.5 = model is uncertain")

#### Custom Patient Predictions — Production Inference

Simulates calling the trained model on brand new patient records.

Java analogy:
```java
double[] features = {6, 148, 72, 35, 0, 33.6, 0.627, 50};
double   prob     = diabetesModel.predict(features);
String   result   = prob >= 0.5 ? "Diabetic" : "Non-Diabetic";
```

**Feature risk reference:**

| Feature | Low Risk | High Risk |
|---|---|---|
| Glucose | < 100 | > 140 (strong signal) |
| BMI | < 25 | > 30 (obese) |
| DiabetesPedigreeFunction | < 0.2 | > 0.6 (family history) |
| Age | < 30 | > 45 |

In [ ]:
# [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DPF, Age]
custom_samples = [
    [6,  148, 72, 35,   0, 33.6, 0.627, 50],  # A: high glucose, older, high DPF  → Diabetic
    [1,   85, 66, 29,   0, 26.6, 0.351, 31],  # B: low glucose, young, normal BMI → Non-Diabetic
    [8,  183, 64,  0,   0, 23.3, 0.672, 32],  # C: very high glucose, many preg   → Diabetic
    [0,  137, 40, 35, 168, 43.1, 2.288, 33],  # D: obese, high insulin, high DPF  → Diabetic
    [5,  116, 74,  0,   0, 25.6, 0.201, 30],  # E: borderline — expect ~50%
    [3,   78, 50, 32,  88, 31.0, 0.248, 26],  # F: young, low glucose             → Non-Diabetic
]
labels = ["Patient A", "Patient B", "Patient C", "Patient D", "Patient E", "Patient F"]

custom_preds = model.predict(np.array(custom_samples))

print(f"{'Patient':<12}  {'Probability':>12}    {'Diagnosis':<20}  Risk")
print("-" * 65)
for label, pred in zip(labels, custom_preds):
    prob = pred[0]
    diag = "Diabetic (1)" if prob >= 0.5 else "Non-Diabetic (0)"
    risk = "HIGH" if prob >= 0.8 else "MODERATE" if prob >= 0.5 else "BORDERLINE" if prob >= 0.3 else "LOW"
    print(f"{label:<12}  {prob:>12.4f}    {diag:<20}  {risk}")

#### Training History Visualization

**What to look for in the plots:**

Loss plot (left):

| Pattern | Meaning |
|---|---|
| Both lines trend down | Model is learning correctly |
| Val loss rises, train loss falls | **Overfitting** |
| Both plateau early | **Underfitting** |

Accuracy plot (right): same logic — large gap between train/val = overfitting.

**Fixing overfitting:** Dropout, L1/L2 regularization, early stopping, more data, simpler model.
**Fixing underfitting:** More neurons/layers, more epochs, lower learning rate.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training History — Diabetes Neural Network", fontsize=13, fontweight="bold")

# Loss plot
axes[0].plot(history.history["loss"],     label="Train Loss",  color="royalblue",  linewidth=2)
axes[0].plot(history.history["val_loss"], label="Val Loss",    color="darkorange", linewidth=2, linestyle="--")
axes[0].set_title("Loss (lower = better)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history.history["accuracy"],     label="Train Accuracy", color="seagreen", linewidth=2)
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy",   color="crimson",  linewidth=2, linestyle="--")
axes[1].set_title("Accuracy (higher = better)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = history.history['val_accuracy'].index(max(history.history['val_accuracy'])) + 1
print(f"Best Val Accuracy: {max(history.history['val_accuracy'])*100:.2f}% at epoch {best_epoch}")
print(f"Tip: Use EarlyStopping(patience=10, restore_best_weights=True) to auto-stop at best epoch")

---

#### Interview Q&A — Deep Learning

---

**Fundamentals**

**Q1: What is the difference between machine learning and deep learning?**
ML uses hand-crafted features with simpler models (logistic regression, random forests). Deep learning automatically learns features through multiple transformation layers. DL excels with unstructured data (images, text, audio); classical ML often wins on small tabular datasets.

**Q2: What is backpropagation?**
After a forward pass computes predictions and loss, backpropagation computes the gradient of the loss with respect to every weight using the chain rule of calculus. These gradients flow backward through the layers. The optimizer then uses them to update weights in the direction that reduces loss.

**Q3: What is the role of an activation function?**
Activations introduce non-linearity. Without them, stacking layers is mathematically equivalent to one linear transformation — the network cannot learn complex patterns. ReLU, Sigmoid, and Softmax are the most common.

**Q4: Explain the bias-variance tradeoff.**
High bias (underfitting) = model too simple, misses patterns. High variance (overfitting) = model too complex, memorizes training data, fails on new data. Goal: balance both so train and val error are both acceptably low.

---

**Architecture**

**Q5: How do you choose the number of layers and neurons?**
There is no formula — it is empirical. Start simple (1-2 hidden layers). Use validation performance to guide tuning. Too many neurons → overfitting; too few → underfitting.

**Q6: Why not use Sigmoid in hidden layers?**
Sigmoid's derivative is always < 0.25, so gradients shrink exponentially through many layers (vanishing gradient). Early layers stop learning. ReLU avoids this — its gradient is 0 or 1 on the positive side.

**Q7: What is Dropout?**
Randomly zeros out a fraction of neurons each training step. Forces the network to learn redundant representations → reduces overfitting. Example: `Dropout(0.2)` after a Dense layer disables 20% of neurons randomly each batch.

---

**Training Mechanics**

**Q8: What is the learning rate?**
Controls step size when updating weights. Too high → overshoots minimum, training diverges. Too low → very slow convergence. Adam adapts the rate per weight automatically.

**Q9: Epoch vs batch?**
Batch = subset of samples processed in one forward+backward pass → one weight update. Epoch = one full pass through all training data. 1000 samples + batch_size=10 → 100 weight updates per epoch.

**Q10: What is Early Stopping?**
Monitors validation loss during training and stops when it stops improving. Saves the best weights automatically.
```python
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(..., callbacks=[es])
```

---

**Practical**

**Q11: fit() vs evaluate() vs predict()?**
- `fit()` — trains: forward + backprop + weight updates
- `evaluate()` — scores: forward only, computes loss + metrics, no weight updates
- `predict()` — infers: forward only, returns raw output, no metrics

**Q12: Why scale features?**
Gradient descent is sensitive to scale. If one feature ranges 0-1000 and another 0-1, gradients are disproportionate → slow or unstable training. StandardScaler or MinMaxScaler ensures all features contribute equally.

**Q13: How would you improve this model?**
1. Feature scaling (StandardScaler) — most impactful step
2. Replace 0-values in Glucose/BP/Insulin with column medians
3. Add `Dropout(0.2)` after hidden layers
4. Add `BatchNormalization()` for faster, more stable training
5. Use `class_weight` in `model.fit()` for class imbalance
6. Early Stopping with `restore_best_weights=True`

---

**Quick Reference**

| Concept | Key Point |
|---|---|
| Dense Layer | Fully connected — every neuron to every neuron |
| ReLU | `max(0,x)` — hidden layers, avoids vanishing gradient |
| Sigmoid | `1/(1+e⁻ˣ)` — binary output, probability [0,1] |
| Binary Cross-Entropy | Loss function for 0/1 classification |
| Adam | Adaptive optimizer — default best choice |
| Overfitting | Train good, val bad → Dropout, regularization |
| Underfitting | Both bad → more neurons, epochs, complexity |
| Backpropagation | Chain rule — gradient flows back to update weights |

---
*End of Notebook — Week 6 Deep Learning*